In [2]:
"""
Задание 2. Реализация внешней сортировки (External Merge Sort)
Суть задания:
В рамках этого задания мы пишем фундамент, на котором работают 
процессы сортировки (и ORDER BY / GROUP BY запросы) в любой 
реляционной БД. Задача: отсортировать файл, размер которого 
превышает доступный лимит оперативной памяти. Поскольку память
ограничена, мы вынуждены использовать диск, но, памятуя о 
результатах первого бенчмарка, мы должны минимизировать 
случайный доступ и обращаться к диску исключительно последовательно.

Что происходит в коде:
Алгоритм реализует парадигму «Разделяй и властвуй» (Divide and Conquer) 
с учетом жестких I/O ограничений. Он делится на две классические фазы:

Фаза 1: create_runs (Генерация прогонов). Мы читаем гигантский исходный 
файл кусками. В нашем скрипте лимит установлен искусственно — 10 МБ (CHUNK_SIZE). Считанный кусок помещается в RAM, поэтому мы имеем право использовать быструю in-memory сортировку (в Python это встроенный Timsort через list.sort()). Отсортированный кусок сохраняется на диск как временный файл (run). В итоге вместо одного неструктурированного хаоса мы получаем 
N отсортированных файлов.
Фаза 2: k_way_merge (K-путевое слияние). Это самая архитектурно важная часть. Нам нужно слить 
N временных файлов в один итоговый. Открывать их по очереди и сравнивать — слишком дорого (потребовалось бы O(N)
 сравнений на каждую строчку). Поэтому мы используем структуру данных Min-Heap (Минимальная куча/Пирамида).
Мы читаем по первой (самой маленькой) строке из каждого временного файла и кладем их в кучу.
Куча автоматически поддерживает наименьший элемент на самом верху (за O(1)).
Мы достаем этот минимум, пишем его в итоговый отсортированный файл на диск.
Затем смотрим, из какого временного файла пришел этот минимум, читаем 
оттуда следующую строку и кладем ее в кучу (восстановление свойства кучи стоит всего 
O(log⁡N)
).
Что требуется сделать студенту (разбор TODO):

В TODO 2.1 и 2.2 нужно отсортировать считанный массив строк chunk и 
записать его во временный файл run_filename. Это закрепит понимание того, как создаются промежуточные результаты (runs).
В TODO 2.3 студент должен инициализировать Min-Heap. Главный подвох 
в том, что в кучу нужно класть не просто строчку line, а кортеж (line, idx). 
Если не сохранить idx (индекс файлового дескриптора), мы на следующем шаге не
узнаем, из какого файла нужно дочитывать новые данные.
В TODO 2.4 реализуется ядро процесса слияния. Нужно корректно использовать heapq.heappop для извлечения 
минимального кортежа, записать извлеченную строку в out_f, после чего обратиться к file_handles[idx], 
прочитать оттуда новую строчку и через heapq.heappush отправить её обратно в кучу (если файл не закончился).
Ожидаемый результат: На выходе получается единый отсортированный output_file.txt. Студент должен увидеть, 
что пиковое потребление RAM за все время работы скрипта не превышало CHUNK_SIZE плюс минимальный overhead 
на работу с кучей и буферы I/O.
"""

'\nЗадание 2. Реализация внешней сортировки (External Merge Sort)\nСуть задания:\nВ рамках этого задания мы пишем фундамент, на котором работают \nпроцессы сортировки (и ORDER BY / GROUP BY запросы) в любой \nреляционной БД. Задача: отсортировать файл, размер которого \nпревышает доступный лимит оперативной памяти. Поскольку память\nограничена, мы вынуждены использовать диск, но, памятуя о \nрезультатах первого бенчмарка, мы должны минимизировать \nслучайный доступ и обращаться к диску исключительно последовательно.\n\nЧто происходит в коде:\nАлгоритм реализует парадигму «Разделяй и властвуй» (Divide and Conquer) \nс учетом жестких I/O ограничений. Он делится на две классические фазы:\n\nФаза 1: create_runs (Генерация прогонов). Мы читаем гигантский исходный \nфайл кусками. В нашем скрипте лимит установлен искусственно — 10 МБ (CHUNK_SIZE). Считанный кусок помещается в RAM, поэтому мы имеем право использовать быструю in-memory сортировку (в Python это встроенный Timsort через list.sort

In [1]:
import os
import heapq

CHUNK_SIZE = 10 * 1024 * 1024  # 10 MB - искусственно ограничиваем "RAM"

def create_runs(input_file, run_prefix="run_"):
    """Фаза 1: Разбиваем исходный массив на куски, сортируем их в RAM и пишем на диск."""
    run_files = []
    
    # Открываем исходный файл на чтение. Для примера считаем, что там лежат строки
    with open(input_file, 'r') as f:
        chunk = f.readlines(CHUNK_SIZE)
        run_idx = 0
        
        while chunk:
            # TODO 2.1: Отсортируйте chunk (напоминаю, это обычная in-memory сортировка)
            
            run_filename = f"{run_prefix}{run_idx}.txt"
            with open(run_filename, 'w') as run_f:
                # TODO 2.2: Запишите отсортированный массив строк в файл run_filename
                pass
                
            run_files.append(run_filename)
            run_idx += 1
            chunk = f.readlines(CHUNK_SIZE)
            
    print(f"Создано отсортированных блоков (runs): {len(run_files)}")
    return run_files

def k_way_merge(run_files, output_file):
    """Фаза 2: Слияние K отсортированных файлов с использованием Min-Heap."""
    # Список файловых дескрипторов, чтобы не закрывать файлы во время чтения
    file_handles = [open(f, 'r') for f in run_files]
    
    # Куча будет хранить кортежи: (строка_данных, индекс_файла_источника)
    min_heap = []
    
    # 1. Инициализация кучи первыми элементами из каждого файла
    for idx, f in enumerate(file_handles):
        line = f.readline()
        if line:
            # TODO 2.3: Положите элемент в min_heap с помощью heapq.heappush
            # ВАЖНО: нужно хранить и саму строку, и 'idx', чтобы знать, откуда брать следующий элемент
            pass

    # 2. Основной цикл слияния
    with open(output_file, 'w') as out_f:
        while min_heap:
            # TODO 2.4: 
            # - Извлеките минимальный элемент из кучи (heapq.heappop)
            # - Запишите извлеченную строку в out_f
            # - Прочтите следующую строку (.readline()) из файла, индекс которого мы извлекли
            # - Если строка не пустая, положите её в кучу
            pass

    # Освобождаем ресурсы
    for f in file_handles:
        f.close()
    
    # Удаляем временные файлы
    for f in run_files:
        os.remove(f)
    print("Внешняя сортировка успешно завершена.")

# Пример использования (заглушка):
# if __name__ == "__main__":
#     runs = create_runs("huge_messy_log.txt")
#     k_way_merge(runs, "sorted_log.txt")
